# SafeRoute Modeling — Data Exploration

Explorasi data scraped: Kaggle flood dataset, OSM roads/waterways/drainage, BMKG weather.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path('../data/raw')
PROC = Path('../data/processed')

## 1. Kaggle Flood Dataset

In [ ]:
df = pd.read_csv(RAW / 'flood_kaggle' / 'flood.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.describe()

In [ ]:
# Flood probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['FloodProbability'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Flood Probability Distribution')
axes[0].set_xlabel('FloodProbability')
axes[0].set_ylabel('Count')

# Top features correlation with flood probability
corr = df.corr(numeric_only=True)['FloodProbability'].drop('FloodProbability').sort_values(ascending=False)
corr.head(10).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 10 Features Correlated with FloodProbability')
axes[1].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

## 2. OSM Roads Network

In [ ]:
with open(PROC / 'roads_graph.json') as f:
    roads = json.load(f)

print(f'Total road segments: {len(roads)}')

highway_types = {}
for r in roads:
    ht = r.get('highway', 'unknown')
    highway_types[ht] = highway_types.get(ht, 0) + 1

print(f'Highway types: {highway_types}')

# Plot road network
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
for road in roads[:2000]:
    coords = road['coords']
    if len(coords) >= 2:
        lats = [c[0] for c in coords]
        lngs = [c[1] for c in coords]
        ax.plot(lngs, lats, '-', linewidth=0.5, color='gray', alpha=0.3)

ax.set_title(f'Semarang Road Network ({len(roads)} segments)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 3. OSM Waterways & Drainage

In [ ]:
with open(PROC / 'waterways_graph.json') as f:
    waterways = json.load(f)

with open(PROC / 'drainage_graph.json') as f:
    drainage = json.load(f)

print(f'Waterways: {len(waterways)} segments')
print(f'Drainage: {len(drainage)} segments')

ww_types = {}
for w in waterways:
    t = w.get('waterway', 'unknown')
    ww_types[t] = ww_types.get(t, 0) + 1
print(f'Waterway types: {ww_types}')

# Plot waterways + drainage
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

for w in waterways:
    coords = w['coords']
    if len(coords) >= 2:
        lats = [c[0] for c in coords]
        lngs = [c[1] for c in coords]
        ax.plot(lngs, lats, '-', linewidth=1, color='blue', alpha=0.5)

for d in drainage:
    coords = d['coords']
    if len(coords) >= 2:
        lats = [c[0] for c in coords]
        lngs = [c[1] for c in coords]
        ax.plot(lngs, lats, '-', linewidth=0.8, color='cyan', alpha=0.4)

ax.set_title(f'Semarang Waterways (blue) & Drainage (cyan)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 4. BMKG Weather

In [ ]:
with open(PROC / 'weather_current.json') as f:
    weather = json.load(f)

print(json.dumps(weather, indent=2))

## 5. Route Engine Test

In [ ]:
from route_engine import calculate_safe_routes
from evacuation_finder import find_nearest_evacuation

# Test: Tembalang -> Stasiun Tawang
flood_zones = [
    {'lat': -6.9535, 'lng': 110.4570, 'radius_km': 1.5, 'status': 'impassable', 'depth_cm': 60},
    {'lat': -6.9620, 'lng': 110.4735, 'radius_km': 1.0, 'status': 'flooded', 'depth_cm': 35},
    {'lat': -6.9450, 'lng': 110.4350, 'radius_km': 1.2, 'status': 'flooded', 'depth_cm': 32},
]

result = calculate_safe_routes(
    origin_lat=-7.0505, origin_lng=110.4410,
    dest_lat=-6.9644, dest_lng=110.4281,
    flood_zones=flood_zones,
    vehicle_max_depth_cm=30,
)

print(f"Flood zones active: {result['flood_zones_active']}")
for opt in result['options']:
    print(f"\n{opt['title']} ({opt['badge']})")
    print(f"  Duration: {opt['duration']} | Distance: {opt['distance']}")
    print(f"  Risk: {opt['risk_level']} | Flood avoided: {opt['flood_avoided']}")

evac = find_nearest_evacuation(-7.0505, 110.4410)
if evac:
    print(f"\nNearest evacuation: {evac['name']}")
    print(f"  Distance: {evac['distance_km']}km | Walk: {evac['duration_walk']}")

## 6. CV Model Architecture

In [ ]:
import torch
from cv_model import FloodClassifier, depth_to_classification

model = FloodClassifier(num_classes=2, pretrained=False, backbone='mobilenet_v3_small')
model.eval()

dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = model(dummy)

probs = torch.softmax(out['logits'], dim=1).numpy()[0]
depth = float(out['depth_cm'].numpy()[0])

print(f"Backbone: mobilenet_v3_small")
print(f"Output: logits={out['logits'].shape}, depth={out['depth_cm'].shape}")
print(f"Probabilities: no_flood={probs[0]:.4f}, flood={probs[1]:.4f}")
print(f"Depth estimate: {depth:.1f}cm -> {depth_to_classification(depth)}")

## 7. Classification Mapping

In [ ]:
from classifier import classify_flood

test_depths = [5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70, 80]

print(f"{'Depth (cm)':>12} | {'Classification':>15} | {'Status':>15} | {'Notification'}")
print('-' * 90)
for depth in test_depths:
    r = classify_flood(depth, 'Kaligawe, Genuk')
    print(f"{depth:>12} | {r.classification:>15} | {r.status_label:>15} | {r.notification}")